In [14]:
!pip install -qU chromadb openai pypdf2 python-docx python-multipart sentence-transformers PyPDF2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

# Document processing and indexing




## document loading

In [15]:
import docx
import PyPDF2
import os
def read_text_file(file_path: str):
  with open(file_path, 'r') as f:
    return f.read()

def read_pdf_file(file_path: str):
  text=""
  with open(file_path, 'rb') as f:
    pdf_reader = PyPDF2.PdfReader(f)
    for page in pdf_reader.pages:
      text += page.extract_text() + "\n"
  return text

def read_docx_file(file_path: str):
  doc=docx.Document(file_path)
  return "\n".join([paragraph.text for paragraph in doc.paragraphs])


## create a unified interface for document reading

In [16]:
def read_document(file_path: str):
    """Read document content based on file extension"""
    _, file_extension = os.path.splitext(file_path)
    file_extension = file_extension.lower()

    if file_extension == '.txt':
        return read_text_file(file_path)
    elif file_extension == '.pdf':
        return read_pdf_file(file_path)
    elif file_extension == '.docx':
        return read_docx_file(file_path)
    else:
        raise ValueError(f"Unsupported file format: {file_extension}")

text=read_document("/content/docs/GreenGrow Innovations_ Company History.docx")
print(text)


GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.

In its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.

By 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their second major product, the SoilHealth Monitor, which used advanced sensors to analyze soil composition and provide real-time recommendations for optimal crop growth.

The company's breakthrough 

## text chunking strategy

In [17]:
def split_text(text: str, chunk_size: int = 500):
    """Split text into chunks while preserving sentence boundaries"""
    sentences = text.replace('\n', ' ').split('. ')
    chunks = []
    current_chunk = []
    current_size = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        # Ensure proper sentence ending
        if not sentence.endswith('.'):
            sentence += '.'

        sentence_size = len(sentence)

        # Check if adding this sentence would exceed chunk size
        if current_size + sentence_size > chunk_size and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentence]
            current_size = sentence_size
        else:
            current_chunk.append(sentence)
            current_size += sentence_size

    # Add the last chunk if it exists
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks


In [18]:
chunks = split_text(text)
print(chunks[0])
len(chunks)

GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient. In its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture.


5

## Setting up chroma DB

In [19]:
import chromadb
from chromadb.utils import embedding_functions
#import textwrap

In [20]:
# Initialize ChromaDB client with persistence
client = chromadb.PersistentClient(path="chroma_db")

# Configure sentence transformer embeddings
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create or get existing collection
collection = client.get_or_create_collection(
    name="documents_collection",
    embedding_function=sentence_transformer_ef
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Inserting data into ChromaDB

In [21]:
def process_document(file_path: str):
    """Process a single document and prepare it for ChromaDB"""
    try:
        # Read the document
        content = read_document(file_path)

        # Split into chunks
        chunks = split_text(content)

        # Prepare metadata
        file_name = os.path.basename(file_path)
        metadatas = [{"source": file_name, "chunk": i} for i in range(len(chunks))]
        ids = [f"{file_name}_chunk_{i}" for i in range(len(chunks))]

        return ids, chunks, metadatas
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")
        return [], [], []


## Batch processing for multiple documents

In [22]:
def add_to_collection(collection, ids, texts, metadatas):
    """Add documents to collection in batches"""
    if not texts:
        return

    batch_size = 100
    for i in range(0, len(texts), batch_size):
        end_idx = min(i + batch_size, len(texts))
        collection.add(
            documents=texts[i:end_idx],
            metadatas=metadatas[i:end_idx],
            ids=ids[i:end_idx]
        )

def process_and_add_documents(collection, folder_path: str):
    """Process all documents in a folder and add to collection"""
    files = [os.path.join(folder_path, file)
             for file in os.listdir(folder_path)
             if os.path.isfile(os.path.join(folder_path, file))]

    for file_path in files:
        print(f"Processing {os.path.basename(file_path)}...")
        ids, texts, metadatas = process_document(file_path)
        add_to_collection(collection, ids, texts, metadatas)
        print(f"Added {len(texts)} chunks to collection")


#### exemple

In [23]:
# Initialize ChromaDB collection (we'll cover this in detail in the next section)
collection = client.get_or_create_collection(
    name="documents_collection",
    embedding_function=sentence_transformer_ef
)

# Process and add documents from a folder
folder_path = "/content/docs"
process_and_add_documents(collection, folder_path)


Processing Company_ QuantumNext Systems.docx...
Added 2 chunks to collection
Processing Company_ GreenFields BioTech.docx...
Added 2 chunks to collection
Processing GreenGrow Innovations_ Company History.docx...
Added 5 chunks to collection
Processing GreenGrow's EcoHarvest System_ A Revolution in Farming.pdf...
Added 6 chunks to collection
Processing Company_ TechWave Innovations.docx...
Added 1 chunks to collection


## Implementing semantic search

In [24]:
def semantic_search(collection, query: str, n_results: int = 2):
    """Perform semantic search on the collection"""
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results




#### exemple

In [25]:
# Perform a search
query = "When was GreenGrow Innovations founded?"
results = semantic_search(collection, query)
results


{'ids': [['GreenGrow Innovations_ Company History.docx_chunk_0',
   'GreenGrow Innovations_ Company History.docx_chunk_4']],
 'embeddings': None,
 'documents': [['GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient. In its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture.',
   'Despite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions to advance the field of agricultural technology and hosts annual conferences to share knowledge with farmers and other industry professionals.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],

## understanding search results

In [26]:
def print_search_results(results):
    """Print formatted search results"""
    print("\nSearch Results:\n" + "-" * 50)

    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        meta = results['metadatas'][0][i]
        distance = results['distances'][0][i]

        print(f"\nResult {i + 1}")
        print(f"Source: {meta['source']}, Chunk {meta['chunk']}")
        print(f"Distance: {distance}")
        print(f"Content: {doc}\n")
print_search_results(results)


Search Results:
--------------------------------------------------

Result 1
Source: GreenGrow Innovations_ Company History.docx, Chunk 0
Distance: 0.16206514835357666
Content: GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient. In its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture.


Result 2
Source: GreenGrow Innovations_ Company History.docx, Chunk 4
Distance: 0.2962738871574402
Content: Despite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions to advance the field of agricultural technology and hosts annual conferences to share knowledge 

In [27]:
def get_context_with_sources(results):
    """Extract context and source information from search results"""
    # Combine document chunks into a single context
    context = "\n\n".join(results['documents'][0])

    # Format sources with metadata
    sources = [
        f"{meta['source']} (chunk {meta['chunk']})"
        for meta in results['metadatas'][0]
    ]

    return context, sources


context, sources= get_context_with_sources(results)
print(context)

GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient. In its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture.

Despite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions to advance the field of agricultural technology and hosts annual conferences to share knowledge with farmers and other industry professionals.


# Gemini integration

## setting up Gemini

In [28]:
!pip install -q google-genai

In [29]:
from google.colab import userdata
from google import genai

api_key=userdata.get("GEMINI_API_KEY")

# Initialize gemini client
client = genai.Client(api_key=api_key)


### /////exemple

In [30]:
completion= client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="What is Retrieval-Augmented Generation (RAG) in simple words",
    config={
        "system_instruction":"You are a helpful assistant."
    }
)

print(completion.text)

Imagine you are taking an open-book exam. 

Without your textbooks (just relying on your memory), you might guess or get facts wrong. But with your textbooks, you can look up the exact information and write a much better, more accurate answer.

**Retrieval-Augmented Generation (RAG)** is basically doing the exact same thing for Artificial Intelligence. 

Here is how it works in two simple steps:

1. **Retrieval:** When you ask an AI a question, it doesn't just guess the answer from its built-in memory. Instead, it first searches a specific set of documents, a database, or the internet to find the exact information needed.
2. **Generation:** The AI takes that retrieved information and uses it to "generate" a clear, accurate, and natural-sounding response for you.

### Why is RAG useful?
* **No outdated info:** Regular AI is stuck with the information it was trained on. RAG allows the AI to look at today's news or your company's latest documents.
* **Fewer "hallucinations":** Because the

## Prompt engineering

In [31]:
def get_prompt(context: str, conversation_history: str, query: str):
    """Generate a prompt combining context, history, and query"""
    prompt = f"""Based on the following context and conversation history,
    please provide a relevant and contextual response. If the answer cannot
    be derived from the context, only use the conversation history or say
    "I cannot answer this based on the provided information."

    Context from documents:
    {context}

    Previous conversation:
    {conversation_history}

    Human: {query}

    Assistant:"""

    return prompt


## response generation

In [32]:
def generate_response(query: str, context: str, conversation_history: str = ""):
    """Generate a response using Gemini with conversation history"""

    prompt = get_prompt(context, conversation_history, query)

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt,
            config={
                "system_instruction": "You are a helpful assistant that answers questions based on the provided context.",
                "temperature": 0,
                "max_output_tokens": 500
            }
        )

        return response.text

    except Exception as e:
        return f"Error generating response: {str(e)}"


## perform RAG query

In [33]:
def rag_query(collection, query: str, n_chunks: int = 2):
    """Perform RAG query: retrieve relevant chunks and generate answer"""
    # Get relevant chunks
    results = semantic_search(collection, query, n_chunks)
    context, sources = get_context_with_sources(results)

    # Generate response
    response = generate_response(query, context)

    return response, sources


In [34]:
query = "When was GreenGrow Innovations founded?"
response, sources = rag_query(collection, query)

# Print results
print("\nQuery:", query)
print("\nAnswer:", response)
print("\nSources used:")
for source in sources:
    print(f"- {source}")



Query: When was GreenGrow Innovations founded?

Answer: GreenGrow Innovations was founded in 2010.

Sources used:
- GreenGrow Innovations_ Company History.docx (chunk 0)
- GreenGrow Innovations_ Company History.docx (chunk 4)


# Building conversational memory

### session management

In [35]:
import uuid
from datetime import datetime
import json

# In-memory conversation store
conversations = {}

def create_session():
    """Create a new conversation session"""
    session_id = str(uuid.uuid4())
    conversations[session_id] = []
    return session_id


### message management

In [36]:
def add_message(session_id: str, role: str, content: str):
    """Add a message to the conversation history"""
    if session_id not in conversations:
        conversations[session_id] = []

    conversations[session_id].append({
        "role": role,
        "content": content,
        "timestamp": datetime.now().isoformat()
    })

def get_conversation_history(session_id: str, max_messages: int = None):
    """Get conversation history for a session"""
    if session_id not in conversations:
        return []

    history = conversations[session_id]
    if max_messages:
        history = history[-max_messages:]

    return history


###formatting conversation history

In [37]:
def format_history_for_prompt(session_id: str, max_messages: int = 5):
    """Format conversation history for inclusion in prompts"""
    history = get_conversation_history(session_id, max_messages)
    formatted_history = ""

    for msg in history:
        role = "Human" if msg["role"] == "user" else "Assistant"
        formatted_history += f"{role}: {msg['content']}\n\n"

    return formatted_history.strip()


### query contextualization

In [49]:
def contextualize_query(query: str, conversation_history: str, client):
    """Convert follow-up questions into standalone queries"""

    system_instruction_for_contextualization = """Given a chat history and the latest user question
    which might reference context in the chat history, formulate a standalone
    question which can be understood without the chat history. Do NOT answer
    the question, just reformulate it if needed and otherwise return it as is."""

    user_query_for_contextualization = f"""Chat history:\n{conversation_history}\n\nQuestion:\n{query}"""

    try:
        completion = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=user_query_for_contextualization,
            config={
                "system_instruction": system_instruction_for_contextualization
            }
        )
        return completion.text
    except Exception as e:
        print(f"Error contextualizing query: {str(e)}")
        return query  # Fallback to original query

# combining RAG components

In [50]:
def get_prompt(context, conversation_history, query):
  prompt = f"""Based on the following context and conversation history, please provide a relevant and contextual response.
    If the answer cannot be derived from the context, only use the conversation history or say "I cannot answer this based on the provided information."

    Context from documents:
    {context}

    Previous conversation:
    {conversation_history}

    Human: {query}

    Assistant:"""
  return prompt


In [51]:
def generate_response(query: str, context: str, conversation_history: str = ""):
    """Generate a response using Gemini with conversation history"""

    prompt = get_prompt(context, conversation_history, query)

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt,
            config={
                "system_instruction": "You are a helpful assistant that answers questions based on the provided context.",
                "temperature": 0,
                "max_output_tokens": 500
            }
        )

        return response.text

    except Exception as e:
        return f"Error generating response: {str(e)}"

In [52]:
def conversational_rag_query(
    collection,
    query: str,
    session_id: str,
    n_chunks: int = 3
):
    """Perform RAG query with conversation history"""
    # Get conversation history
    conversation_history = format_history_for_prompt(session_id)

    # Handle follo up questions
    query = contextualize_query(query, conversation_history, client)
    print("Contextualized Query:", query)

    # Get relevant chunks
    context, sources = get_context_with_sources(
        semantic_search(collection, query, n_chunks)
    )
    print("Context:", context)
    print("Sources:", sources)


    response = generate_response(query, context, conversation_history)

    # Add to conversation history
    add_message(session_id, "user", query)
    add_message(session_id, "assistant", response)

    return response, sources


## conversation 1

In [53]:
session_id= create_session()
print(session_id)

da233f0b-9cbb-41da-8a76-059b786d6c98


In [54]:
# First question
query = "When was GreenGrow Innovations founded?"
response, sources = conversational_rag_query(
            collection,
            query,
            session_id
)
print(response)

Contextualized Query: When was GreenGrow Innovations founded?
Context: GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient. In its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture.

Despite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions to advance the field of agricultural technology and hosts annual conferences to share knowledge with farmers and other industry professionals.

This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence. Today, GreenGrow Innovations employs over

In [55]:
query = "Where is it located?"
response, sources = conversational_rag_query(
            collection,
            query,
            session_id
)
print(response)


Contextualized Query: Where is GreenGrow Innovations located?
Context: GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient. In its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture.

This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence. Today, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.

Despite its growth, GreenGrow remains commit